# Course 3 lab — Evidence-bound governance operating model

**Scenario:** a procurement agent searches approved vendors, emails suppliers, and creates purchase orders up to USD 25,000.

**Outcome:** build an inventory, route specialist reviews, select controls, validate version-bound evidence, make an internal release-gate decision, detect material change, and generate a machine-readable governance package.

**Boundary:** this offline lab routes regulatory questions to specialists. It does not make legal determinations, certify an organization, authorize a deployment, or prove compliance.


![Standards and regulation landscape](assets/01-standards-regulation-landscape.svg)

NIST AI RMF and ISO/IEC 23894 structure risk management; ISO/IEC 42001 structures an organization-wide management system; ISO/IEC 42005 focuses impact assessment; regulation creates legal duties; OWASP supplies agent-security guidance; OSCAL supplies machine-readable control models.


## 1. Offline setup

The notebook imports the same `lab.py` exercised by tests. It makes no network calls, installs no packages, reads no credentials, and writes no files. A fixed 20 September 2026 snapshot makes time checks reproducible.


In [ ]:
from datetime import date
from pathlib import Path
from pprint import pprint
import sys
from jsonschema import validate

LAB_DIR = Path("curriculum/beginner/03-standards-regulation-and-governance-operating-model").resolve()
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

from lab import (
    CONTROL_PROFILE_VERSION, METHODOLOGY_SNAPSHOT, AgentSystemRecord,
    AutonomyLevel, EvidenceResult, ExceptionRecord, GateOutcome, GateRequest,
    GOVERNANCE_PACKAGE_SCHEMA, RACIEntry, ReviewDomain, assess_change,
    assess_evidence, build_applicability_record, build_demo_package,
    control_library, demo_completed_reviews, demo_evidence, demo_raci,
    demo_system, evaluate_exception, evaluate_gate, evaluation_report,
    oscal_component_projection, pending_review_domains, required_review_domains,
    select_controls, snapshot, system_digest, system_tags, validate_raci,
)
AS_OF = METHODOLOGY_SNAPSHOT
print("Methodology snapshot:", AS_OF)


## 2. Baseline: percentage-complete approval is unsafe

Coverage belongs on a dashboard, but an average is not a release rule. If the one missing item is authorization evidence, 92% complete is not “almost approved.”


In [ ]:
required, present = 12, 11
print("Naive completeness:", f"{present / required:.1%}")
print("Decision: UNKNOWN — inspect the named missing requirement")


## 3. Build a versioned inventory

The record contains intended and prohibited use, owners, capabilities, data, jurisdictions, affected groups, autonomy, and internal risk. A digest binds later evidence and decisions to this exact snapshot.


In [ ]:
system = demo_system()
print(system.system_id, system.version)
print("System digest:", system_digest(system))
print("Applicability tags:", sorted(system_tags(system)))
for capability in system.capabilities:
    pprint(capability.model_dump(mode="json"))


## 4. Select controls and keep crosswalk claims narrow

Typed capability facts select controls. Every external mapping says an internal control **supports** a framework theme; it does not say “equivalent,” “certified,” or “compliant.”


In [ ]:
controls = select_controls(system, control_library())
for control in controls:
    print(f"{control.control_id}: {control.title} — {control.owner_role}")
    for mapping in control.mappings:
        print("  ", mapping.relationship, mapping.framework, "->", mapping.reference)


## 5. Route regulatory questions to specialists

Application code may identify review triggers. It must not infer provider/deployer status, prohibited practice, Annex classification, or conformity duties from a few inventory fields. This experiment adds an EU deployment but deliberately omits the EU AI Act review.


In [ ]:
eu_system = demo_system(include_eu=True)
eu_applicability = build_applicability_record(
    eu_system,
    completed_by={
        ReviewDomain.SECURITY: "Security Architecture reviewer",
        ReviewDomain.IMPACT_ASSESSMENT: "Impact reviewer",
    },
    as_of=AS_OF,
)
print("Required:", [item.value for item in required_review_domains(eu_system)])
print("Pending:", [item.value for item in pending_review_domains(eu_applicability)])
print(eu_applicability.disclaimer)


## 6. Validate RACI as data

Each activity needs exactly one accountable role and at least one responsible role. This catches structural gaps; it cannot by itself prove real authority, capacity, or independence.


In [ ]:
raci = demo_raci()
print(validate_raci(raci, [item.activity for item in raci]))
try:
    RACIEntry(activity="Release", accountable=("Business", "Governance"), responsible=("Engineering",))
except ValueError as error:
    print("Failure injection caught:", str(error).splitlines()[0])


## 7. Require scoped, current, passing evidence

An evidence ID is not proof. Each artifact binds to system/version, control/requirement, result, digest, dates, producer, and environment.


In [ ]:
evidence = demo_evidence(system, controls, as_of=AS_OF)
assessment = assess_evidence(system, controls, evidence, as_of=AS_OF)
pprint(assessment.model_dump(mode="json"))
assert assessment.required_count == assessment.satisfied_count


### Failure injection: wrong version and failed result


In [ ]:
corrupted = list(evidence)
corrupted[0] = corrupted[0].model_copy(update={"system_version": "0.9.0"})
corrupted[1] = corrupted[1].model_copy(update={"result": EvidenceResult.FAIL})
failed = assess_evidence(system, controls, corrupted, as_of=AS_OF)
print("Missing:", failed.missing_requirements)
print("Rejected:", failed.rejected_evidence)


## 8. Make an exact internal release-gate decision

The gate binds the request to the system digest and control-profile version. All routed reviews and named evidence requirements must pass. Its legal-compliance field is structurally fixed to false.


In [ ]:
package = build_demo_package(as_of=AS_OF)
payload = package.model_dump(mode="json")
validate(instance=payload, schema=GOVERNANCE_PACKAGE_SCHEMA)
print("Outcome:", package.decision.outcome.value)
print("Evidence:", package.evidence_assessment.satisfied_count, "/", package.evidence_assessment.required_count)
print("Decision digest:", package.decision.decision_digest)
print("Package digest:", package.package_digest())
print("Legal compliance established:", package.decision.legal_compliance_established)


### Failure injection: one missing item cannot be averaged away

Remove only the independent-assurance report. Nearly every artifact remains, but the gate blocks on that named gap.


In [ ]:
incomplete = tuple(item for item in evidence if item.requirement_id != "independent_assurance_report")
incomplete_assessment = assess_evidence(system, controls, incomplete, as_of=AS_OF)
applicability = build_applicability_record(system, completed_by=demo_completed_reviews(system), as_of=AS_OF)
request = GateRequest(
    request_id="GATE-NOTEBOOK-MISSING", system_id=system.system_id,
    system_version=system.version, system_digest=system_digest(system),
    control_profile_version=CONTROL_PROFILE_VERSION, target_environment="production",
)
blocked = evaluate_gate(request, system, controls, applicability, incomplete_assessment, as_of=AS_OF)
print(blocked.outcome.value, blocked.reason_codes)
assert blocked.outcome is GateOutcome.BLOCKED


## 9. Detect material change

An EU expansion, higher autonomy, and a new affected population invalidate the old package and trigger scoped reassessment.


In [ ]:
after_payload = system.model_dump()
after_payload.update(
    version="2.0.0", jurisdictions=frozenset({"CA", "EU"}),
    autonomy=AutonomyLevel.HIGH,
    affected_groups=("procurement staff", "suppliers", "job applicants"),
)
after = AgentSystemRecord(**after_payload)
change = assess_change(snapshot(system), snapshot(after))
print("Triggers:", [item.value for item in change.triggers])
print("Reassessment:", change.reassessment_scopes)
print("Full reassessment:", change.full_reassessment_required)


## 10. Bound exceptions

Exceptions require separation of requester and risk acceptor, compensating controls, evidence, remediation, and expiry. Runtime authorization is non-exception-eligible in this teaching policy.


In [ ]:
exception = ExceptionRecord(
    exception_id="EX-NOTEBOOK-001", system_id=system.system_id,
    system_version=system.version, control_id="AG-AUTH-001",
    rationale="Authorization migration is incomplete.",
    compensating_controls=("Disable state-changing capabilities",),
    requester="Technical Owner", risk_acceptor="Business Owner",
    issued_on=date(2026, 8, 1), expires_on=date(2026, 9, 1),
    remediation_plan="Complete runtime policy enforcement.",
    evidence_ids=("EV-COMP-001",),
)
print(evaluate_exception(exception, system, controls, as_of=AS_OF))


## 11. Project controls toward OSCAL without false conformance

OSCAL 1.2.3 is the latest official release at this snapshot. Compliance Trestle v5 is actively developed and reports support for OSCAL 1.2.1. This lab produces only an explicitly labelled teaching projection. Production exchange requires the correct OSCAL model and official schema or maintained-tool validation.


In [ ]:
projection = oscal_component_projection(system, controls)
print(projection["format"], "target", projection["target_oscal_version"])
print("Projected controls:", len(projection["component_definition_projection"]))
print(projection["validation_required"])


## 12. Evaluate the gate on a labelled fixture

The fixture covers complete, missing, expired, pending-review, and altered-request cases. A perfect result on five examples proves only that these teaching cases match their labels.


In [ ]:
report = evaluation_report()
pprint(report)
assert report == {"correct": 5, "cases": 5, "decision_accuracy": 1.0}


## 13. Method and tool comparison

| Need | Common method/tool | Strength | Boundary |
|---|---|---|---|
| AI risk | NIST AI RMF; ISO/IEC 23894 | Flexible lifecycle risk outcomes | Version profiles and mappings; AI RMF 1.0 is under revision |
| Management system | ISO/IEC 42001 | Organization-wide AIMS and continual improvement | Certification is scope-specific, not product-safety proof |
| Impact | ISO/IEC 42005 | Lifecycle effects on people, groups, society | Requires context and stakeholder evidence |
| Agent security | OWASP Agentic Top 10; NIST/ATLAS | Threat scenarios and engineering controls | A taxonomy mapping is not effectiveness evidence |
| Accountability | IIA Three Lines; RACI; gates | Ownership, challenge, assurance | A matrix cannot establish actual authority |
| Structured artifacts | Pydantic; JSON Schema; Git/CI | Typed, versioned, testable packages | Schema-valid data may still be false |
| Control exchange | OSCAL; Compliance Trestle | Machine-readable control workflows | Validate real artifacts and compatibility |
| Enterprise workflow | GRC/IRM and AI governance platforms | Portfolio workflow and evidence | Preserve exact scope, version, and provenance |


## 14. Production upgrade and exercises

| Lab | Production upgrade |
|---|---|
| In-memory records | Authenticated tenant-aware registry, optimistic locking, retention, recovery |
| Reviewer string | IdP group/authority checks, separation of duties, immutable audit event |
| Static tags | Versioned policy service with explainable rules and controlled overrides |
| URI + digest | Content-addressed evidence store, signed provenance, access controls |
| Single gate | Transactional workflow, concurrency, expiry, appeal, rollback |
| Static sources | Standards/regulatory change management and recertification triggers |
| Internal schema | Schema registry plus official OSCAL validation when used |

Exercises:

1. Add a low-risk summarizer and compare its controls.
2. Inject future-dated and wrong-version evidence.
3. Add a jurisdiction without inventing a legal conclusion.
4. Design concurrent review and system-update handling.
5. Add wrong-system, failed-test, and altered-profile evaluation cases.

Checkpoint: explain the difference between an internal risk tier, regulatory applicability, a crosswalk, evidence effectiveness, and a release-gate outcome.


## Primary and official references

- [NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)
- [NIST AI RMF Generative AI Profile](https://www.nist.gov/publications/artificial-intelligence-risk-management-framework-generative-artificial-intelligence)
- [NIST AI RMF crosswalks](https://airc.nist.gov/airmf-resources/crosswalks/)
- [ISO/IEC 42001:2023](https://www.iso.org/standard/42001)
- [ISO/IEC 42005:2025](https://www.iso.org/standard/42005)
- [ISO/IEC 23894:2023](https://www.iso.org/standard/77304.html)
- [ISO/IEC 42006:2025](https://www.iso.org/standard/42006)
- [European Commission AI Act overview](https://digital-strategy.ec.europa.eu/en/policies/regulatory-framework-ai)
- [OWASP Top 10 for Agentic Applications 2026](https://genai.owasp.org/resource/owasp-top-10-for-agentic-applications-for-2026/)
- [NIST OSCAL](https://pages.nist.gov/OSCAL/)
- [OSCAL 1.2.3](https://github.com/usnistgov/OSCAL/releases/tag/v1.2.3)
- [Compliance Trestle](https://github.com/oscal-compass/compliance-trestle)
- [IIA Three Lines statements](https://www.theiia.org/en/resources/statements-of-position)
